# Hybrid ETL — At-Risk Student Dashboard

**Development phase pipeline.** Connects to Moodle sandbox, falls back to mock data per cell when real data is unavailable.

## Notebook Structure

1. Pre-flight check on the Moodle token
2. Extract real participants (with role) from `core_enrol_get_enrolled_users`
3. **Layer 1 fallback** — if real cohort < `COHORT_THRESHOLD`, use the 22-student demo cohort
4. Extract real grade items from `gradereport_user_get_grade_items` where available
5. **Layer 2 fallback** — for each missing metric, fill with deterministic mock values
6. Compute risk scores using thresholds locked in `etl/schemas.py`
7. Generate structured tables 
- `fact_student_risk.csv`
- `dim_student.csv`
- `dim_course.csv`
- `fact_assessment.csv`
- `dim_assessment.csv`
   
8. Save all 5 outputs (CSV + Parquet) to `data/star_schema/` to feed into PowerBI

## Power BI Refresh

Refresh the dashboard to new data. 
- Manual refresh(development phase)
- Schdeduled Jobs(production phase in Microsoft Fabric)


## 1. Setup

In [1]:
# =========================
# UNIVERSAL ENVIRONMENT SETUP + ETL IMPORTS
# =========================
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

import numpy as np     
import pandas as pd    


def load_env_safely():
    current_path = Path.cwd()
    for i in range(5):
        env_path = current_path / ".env"
        if env_path.exists():
            print(f"✅ .env found at: {env_path}")
            load_dotenv(env_path, override=True)
            return env_path
        current_path = current_path.parent
    print("❌ .env file NOT found!")
    return None

env_path = load_env_safely()

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Now safe to import etl modules — env vars are loaded
from etl import schemas as S
from etl import io as etl_io
from etl import moodle_client as mc
from etl import transforms as T
from etl import risk as R

# Note: hybrid_etl need mocks
try:
    from etl import mocks as M
except ImportError:
    M = None

print("\n===== ENV STATUS =====")
print("Working directory:", Path.cwd())
print("Base URL:", mc.MOODLE_BASE_URL)
print("Token exists:", bool(mc.MOODLE_TOKEN))
print("Token preview:", mc.MOODLE_TOKEN[:6] + "..." if mc.MOODLE_TOKEN else "EMPTY")
print("Endpoint:", mc.MOODLE_ENDPOINT)
print("======================\n")

✅ .env found at: /Users/Kay/MDS651 Industry Capstone/.env

===== ENV STATUS =====
Working directory: /Users/Kay/MDS651 Industry Capstone/Notebook
Base URL: https://spi.nsw.edu.au/learn
Token exists: True
Token preview: 4daf5d...
Endpoint: https://spi.nsw.edu.au/learn/webservice/rest/server.php



## 2. Configuration

Tune these knobs per environment. None of them affect the dashboard's data contract.

In [2]:
# ============================================================
# Pipeline configuration
# ============================================================

# Cohort threshold: if real participant count < this number, fall back
# to the 22-student demo cohort entirely.
COHORT_THRESHOLD = 10

# Random seed used for ALL deterministic mock generation. Same seed +
# same input -> same output, every run.
BASE_SEED = S.DEFAULT_BASE_SEED  # = 42

# Output directories. 
DATA_DIR = PROJECT_ROOT.parent / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
STAR_SCHEMA_DIR = DATA_DIR / "star_schema"

etl_io.ensure_dirs(DATA_DIR, RAW_DIR, PROCESSED_DIR, STAR_SCHEMA_DIR)
print(f"Star schema output dir: {STAR_SCHEMA_DIR}")


Star schema output dir: /Users/Kay/MDS651 Industry Capstone/data/star_schema


## 3. Pre-flight token check

If this fails, the rest of the pipeline still runs but everything goes through mock fallback. The `data_source` column on every output row will record exactly why.

In [3]:
preflight = mc.preflight_check()
real_extraction_possible = preflight["ok"]

if real_extraction_possible:
    failure_data_source = None
else:
    failure_data_source = {
        "token_invalid": S.DATA_SOURCE_MOCK_TOKEN_INVALID,
        "endpoint_blocked": S.DATA_SOURCE_MOCK_ENDPOINT_BLOCKED,
        "network_error": S.DATA_SOURCE_MOCK_NETWORK_ERROR,
    }.get(preflight["failure_mode"], S.DATA_SOURCE_MOCK_FULL)
    print(f"  -> Will use full mock fallback. data_source = '{failure_data_source}'")


[moodle] Pre-flight FAILED (token_invalid): invalidtoken: Invalid token - token not found
  -> Will use full mock fallback. data_source = 'mock_full_token_invalid'


## 4. Course catalog (real or fallback)

In [4]:
if real_extraction_possible:
    site_info_result = mc.extract_site_info()
    courses_result = mc.extract_all_courses()
    etl_io.save_json(site_info_result, RAW_DIR / "site_info.json")
    etl_io.save_json(courses_result, RAW_DIR / "all_courses.json")

    course_catalog = T.transform_course_catalog(courses_result, site_info_result)
    site_df = T.transform_site_info(site_info_result)

    if course_catalog.empty:
        print("Course catalog empty from API. Using demo course catalog.")
        course_catalog = M.generate_demo_courses()
    else:
        print(f"Real course catalog: {len(course_catalog)} courses")
else:
    site_info_result = {"success": False, "data": None, "error": preflight["error"]}
    courses_result = {"success": False, "data": None, "error": preflight["error"]}
    course_catalog = M.generate_demo_courses()
    site_df = pd.DataFrame()
    print(f"Using demo course catalog: {len(course_catalog)} courses")

display(course_catalog)


Using demo course catalog: 2 courses


,course_id,course_shortname,course_name
0,210,SC-1,Sample Course 1
1,211,SC-2,Sample Course 2


## 5. Cohort identity — Layer 1 fallback

Try to extract real enrolled users (students AND teachers — teachers will be kept in `dim_student` with `role='teacher'`, but excluded from the at-risk fact tables).

In [5]:
real_participants_long = pd.DataFrame()
participant_results = {}
participant_data_source = None

if real_extraction_possible and not course_catalog.empty:
    for cid in course_catalog["course_id"].dropna().astype(int).tolist():
        result = mc.extract_course_participants(cid)
        participant_results[cid] = result
        etl_io.save_json(result, RAW_DIR / f"participants_course_{cid}.json")

    real_participants_long = T.transform_course_participants(participant_results)
    print(f"Real participants extracted: {len(real_participants_long)} rows (across all courses)")

unique_real_participants = (
    real_participants_long["student_id"].nunique() if not real_participants_long.empty else 0
)
print(f"Unique real users: {unique_real_participants}")
print(f"Cohort threshold: {COHORT_THRESHOLD}")

if unique_real_participants >= COHORT_THRESHOLD:
    cohort_long = real_participants_long.copy()
    participant_data_source = S.DATA_SOURCE_MOODLE_FULL
    print(f"  -> Using REAL cohort ({unique_real_participants} users)")
else:
    if real_extraction_possible and unique_real_participants > 0:
        participant_data_source = S.DATA_SOURCE_MOCK_COHORT_TOO_SMALL
        print(f"  -> Real cohort too small. Falling back to demo cohort.")
    else:
        participant_data_source = failure_data_source or S.DATA_SOURCE_MOCK_FULL
        print(f"  -> No real participants. Using demo cohort. data_source = '{participant_data_source}'")

    demo_students = M.generate_demo_cohort()
    demo_students["_k"] = 1
    course_catalog["_k"] = 1
    cohort_long = (
        demo_students.merge(course_catalog, on="_k")
        .drop(columns="_k")
        .assign(lastaccess_unix=np.nan, has_login=0)
    )
    course_catalog = course_catalog.drop(columns="_k")

print(f"Cohort long-format rows: {len(cohort_long)}")


Unique real users: 0
Cohort threshold: 10
  -> No real participants. Using demo cohort. data_source = 'mock_full_token_invalid'
Cohort long-format rows: 44


## 6. Real grade extraction (best-effort)

Tries `gradereport_user_get_grade_items` for each (student × course) pair. Sparse responses are normal — the per-cell fallback fills empty grades with deterministic mocks.

In [6]:
grade_summary_rows = []

if real_extraction_possible and not cohort_long.empty:
    pairs = cohort_long[["student_id", "course_id"]].drop_duplicates()
    print(f"Attempting grade extraction for {len(pairs)} student-course pairs...")
    for _, p in pairs.iterrows():
        result = mc.extract_user_grade_items(int(p["student_id"]), int(p["course_id"]))
        if result.get("success"):
            etl_io.save_json(
                result,
                RAW_DIR / f"grades_user_{int(p['student_id'])}_course_{int(p['course_id'])}.json",
            )
        grade_summary_rows.append(
            T.transform_grade_items_summary(result, int(p["student_id"]), int(p["course_id"]))
        )

if grade_summary_rows:
    grades_df = pd.DataFrame(grade_summary_rows)
    real_count = int(grades_df["real_grade_available"].fillna(False).sum())
    print(f"Real grade items returned for {real_count}/{len(grades_df)} pairs")
else:
    grades_df = pd.DataFrame(columns=[
        "student_id", "course_id", "real_grade_available",
        "avg_grade_pct_real", "missing_grade_count_real", "grade_item_count_real",
    ])
    print("No grade extraction attempted (no real cohort or no token).")


No grade extraction attempted (no real cohort or no token).


## 7. Per-cell fallback (Layer 2)

For each (student × course) row:
- `attendance_pct` — always simulated (no Moodle source until `mod_attendance` is intergrated)
- `avg_grade_pct` — real if extracted, deterministic mock otherwise
- `days_since_last_login` — real from `lastaccess`, BLANK if `lastaccess=0` (fall into category "Never Login" bin)


In [7]:
# Merge grades into the cohort, then enrich
if not grades_df.empty:
    cohort_with_grades = cohort_long.merge(
        grades_df, on=["student_id", "course_id"], how="left"
    )
else:
    cohort_with_grades = cohort_long.assign(
        real_grade_available=False,
        avg_grade_pct_real=np.nan,
        missing_grade_count_real=np.nan,
        grade_item_count_real=0,
    )

cohort_enriched = M.enrich_with_per_cell_fallback(cohort_with_grades, base_seed=BASE_SEED)
print(f"Cohort enriched: {len(cohort_enriched)} rows")


Cohort enriched: 44 rows


## 8. Risk scoring

### Risk thresholds the dashboard depends on:
- Attendance: high <70%, moderate 70-79%, low ≥80%
- Grade: high <50%, moderate 50-64%, low ≥65%
- Login: high >14 days, moderate 7-14 days, low <7 days
- Weights: high=3, moderate=2 → risk_score(0-9).
- Risk Level: High≥7, Medium≥3, Low<3

In [8]:
cohort_risk = R.apply_final_risk_logic(cohort_enriched)

print("Risk score distribution:")
print(cohort_risk["risk_score"].value_counts().sort_index())
print()
print("Risk level distribution:")
print(cohort_risk["risk_level"].value_counts())


Risk score distribution:
risk_score
0     3
2     8
3    10
4     6
5     9
6     3
7     2
8     1
9     2
Name: count, dtype: int64

Risk level distribution:
risk_level
Medium    28
Low       11
High       5
Name: count, dtype: int64


## 9. Build star-schema tables

In [9]:
fact_student_risk = T.build_fact_student_risk(
    cohort_risk,
    data_source=participant_data_source,
    students_only=True,   # excludes teachers from the at-risk fact table
)

# dim_student: all participants including teachers (with role column)
if participant_data_source == S.DATA_SOURCE_MOODLE_FULL:
    dim_student = T.build_dim_student(real_participants_long)
else:
    dim_student = M.generate_demo_cohort()

dim_course = T.build_dim_course(course_catalog)
dim_site = T.build_dim_site(site_df, base_url_fallback=mc.MOODLE_BASE_URL)
dim_date = T.build_dim_date(fact_student_risk)

print(f"dim_student: {len(dim_student)} rows  ({(dim_student['role']=='student').sum()} students, {(dim_student['role'].isin(S.TEACHER_ROLES)).sum()} teachers)")
print(f"dim_course: {len(dim_course)} rows")
print(f"fact_student_risk: {len(fact_student_risk)} rows (students only)")


dim_student: 22 rows  (22 students, 0 teachers)
dim_course: 2 rows
fact_student_risk: 44 rows (students only)


## 10. Assessment-level extension

In [10]:
fact_rows = []
dim_rows = []
assignment_dim_rows = []

if real_extraction_possible:
    for cid in course_catalog["course_id"].dropna().astype(int).unique():
        a_result = mc.extract_assignments_for_course(int(cid))
        etl_io.save_json(a_result, RAW_DIR / f"assignments_course_{cid}.json")
        assignment_dim_rows.extend(T.transform_assignments_to_dim_rows(a_result, int(cid)))

    if not real_participants_long.empty:
        students_only = real_participants_long[real_participants_long["role"] == "student"]
        pairs = students_only[["student_id", "course_id"]].drop_duplicates()
        for _, p in pairs.iterrows():
            g = mc.extract_user_grade_items(int(p["student_id"]), int(p["course_id"]))
            f_rows, d_rows = T.transform_grade_items_to_assessment_rows(
                g, int(p["student_id"]), int(p["course_id"])
            )
            fact_rows.extend(f_rows)
            dim_rows.extend(d_rows)

if not fact_rows:
    print("Real assessment data sparse/empty — using deterministic fallback.")
    fact_assessment_df, dim_assessment_df = M.build_fallback_assessment_tables(
        fact_student_risk, dim_student, dim_course, base_seed=BASE_SEED,
    )
else:
    fact_assessment_df = pd.DataFrame(fact_rows)
    dim_assessment_df = pd.DataFrame(dim_rows + assignment_dim_rows)

fact_assessment = T.build_fact_assessment(fact_assessment_df)
dim_assessment = T.build_dim_assessment(dim_assessment_df)

print(f"dim_assessment: {len(dim_assessment)} rows")
print(f"fact_assessment: {len(fact_assessment)} rows")


Real assessment data sparse/empty — using deterministic fallback.
dim_assessment: 6 rows
fact_assessment: 132 rows


## 11. Save outputs

In [11]:
etl_io.save_dataframe(fact_student_risk, STAR_SCHEMA_DIR / "fact_student_risk")
etl_io.save_dataframe(dim_student, STAR_SCHEMA_DIR / "dim_student")
etl_io.save_dataframe(dim_course, STAR_SCHEMA_DIR / "dim_course")
etl_io.save_dataframe(dim_site, STAR_SCHEMA_DIR / "dim_site")
etl_io.save_dataframe(dim_date, STAR_SCHEMA_DIR / "dim_date")
etl_io.save_dataframe(fact_assessment, STAR_SCHEMA_DIR / "fact_assessment")
etl_io.save_dataframe(dim_assessment, STAR_SCHEMA_DIR / "dim_assessment")

data_source_counts = (
    fact_student_risk["data_source"].value_counts().to_dict()
    if "data_source" in fact_student_risk.columns else {}
)
data_source_summary = ", ".join(f"{k}={v}" for k, v in sorted(data_source_counts.items()))

run_summary = pd.DataFrame([{
    "run_timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "pipeline": "hybrid",
    "site_info_success": preflight.get("ok", False),
    "course_catalog_success": courses_result.get("success", False) if isinstance(courses_result, dict) else False,
    "participant_extraction_mode": participant_data_source,
    "real_participant_count": int(unique_real_participants),
    "student_count": int((dim_student["role"] == "student").sum()),
    "teacher_count": int(dim_student["role"].isin(S.TEACHER_ROLES).sum()),
    "course_count": int(dim_course["course_id"].nunique()),
    "fact_rows": int(len(fact_student_risk)),
    "real_grade_rows": int(grades_df["real_grade_available"].fillna(False).sum()) if not grades_df.empty else 0,
    "risk_low_count": int((fact_student_risk["risk_level"] == "Low").sum()),
    "risk_medium_count": int((fact_student_risk["risk_level"] == "Medium").sum()),
    "risk_high_count": int((fact_student_risk["risk_level"] == "High").sum()),
    "assessment_fact_rows": int(len(fact_assessment)),
    "assessment_count": int(dim_assessment["assessment_id"].nunique()),
    "assessment_data_source": ", ".join(sorted(fact_assessment["data_source"].dropna().unique().tolist())) if not fact_assessment.empty else "none",
    "data_source_summary": data_source_summary,
    "schema_note": "Hybrid ETL — fact_student_risk students-only, dim_student includes teachers with role column",
}])

etl_io.save_dataframe(run_summary, STAR_SCHEMA_DIR / "etl_run_summary")
etl_io.save_dataframe(run_summary, PROCESSED_DIR / "etl_run_summary")

print()
print("=== Run summary ===")
display(run_summary.T)


[io] Saved CSV: fact_student_risk.csv  (44 rows)
[io] Saved Parquet: fact_student_risk.parquet
[io] Saved CSV: dim_student.csv  (22 rows)
[io] Saved Parquet: dim_student.parquet
[io] Saved CSV: dim_course.csv  (2 rows)
[io] Saved Parquet: dim_course.parquet
[io] Saved CSV: dim_site.csv  (1 rows)
[io] Saved Parquet: dim_site.parquet
[io] Saved CSV: dim_date.csv  (1 rows)
[io] Saved Parquet: dim_date.parquet
[io] Saved CSV: fact_assessment.csv  (132 rows)
[io] Saved Parquet: fact_assessment.parquet
[io] Saved CSV: dim_assessment.csv  (6 rows)
[io] Saved Parquet: dim_assessment.parquet
[io] Saved CSV: etl_run_summary.csv  (1 rows)
[io] Saved Parquet: etl_run_summary.parquet
[io] Saved CSV: etl_run_summary.csv  (1 rows)
[io] Saved Parquet: etl_run_summary.parquet

=== Run summary ===


,0
run_timestamp_utc,2026-05-11T00:14:16.636975+00:00
pipeline,hybrid
site_info_success,False
course_catalog_success,False
participant_extraction_mode,mock_full_token_invalid
real_participant_count,0
student_count,22
teacher_count,0
course_count,2
fact_rows,44


## 12. Quick visual check

In [12]:
print("Sample fact_student_risk Table:")
display(fact_student_risk.head(5))
print()
print("data_source distribution:")
display(fact_student_risk["data_source"].value_counts())


Sample fact_student_risk Table:


,student_id,course_id,attendance_pct,avg_grade_pct,grade_item_count,days_since_last_login,risk_high_attendance,risk_moderate_attendance,risk_high_grade,risk_moderate_grade,risk_high_last_login,risk_moderate_last_login,risk_score,risk_level,recommended_action,data_source,etl_loaded_at,etl_load_date
0,1001,210,82.6,76.3,3,19,False,False,False,False,True,False,3,Medium,Monitor closely and send early warning,mock_full_token_invalid,2026-05-11T00:14:16.154666+00:00,2026-05-11
1,1001,211,85.2,50.6,5,3,False,False,False,True,False,False,2,Low,No immediate intervention required,mock_full_token_invalid,2026-05-11T00:14:16.154666+00:00,2026-05-11
2,1002,210,81.5,69.1,2,20,False,False,False,False,True,False,3,Medium,Monitor closely and send early warning,mock_full_token_invalid,2026-05-11T00:14:16.154666+00:00,2026-05-11
3,1002,211,96.3,93.2,4,12,False,False,False,False,False,True,2,Low,No immediate intervention required,mock_full_token_invalid,2026-05-11T00:14:16.154666+00:00,2026-05-11
4,1003,210,83.4,61.8,4,8,False,False,False,True,False,True,4,Medium,Monitor closely and send early warning,mock_full_token_invalid,2026-05-11T00:14:16.154666+00:00,2026-05-11



data_source distribution:


data_source
mock_full_token_invalid    44
Name: count, dtype: int64

In [13]:
print("Sample fact_assessment Table:")
display(fact_assessment.head(10))
print()

Sample fact_assessment Table:


,student_id,course_id,assessment_id,assessment_name,assessment_type,grade,max_grade,grade_pct,submission_status,submitted_at,due_date,days_late,is_late,is_missing,is_below_pass,data_source,etl_loaded_at,etl_load_date
0,1001,210,210_A1,Assignment 1,Assignment,78.74,100,78.74,Late,None,None,12,1,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
1,1002,210,210_A1,Assignment 1,Assignment,60.78,100,60.78,Late,None,None,13,1,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
2,1003,210,210_A1,Assignment 1,Assignment,67.80,100,67.80,Submitted,None,None,0,0,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
3,1004,210,210_A1,Assignment 1,Assignment,91.12,100,91.12,Submitted,None,None,0,0,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
4,1005,210,210_A1,Assignment 1,Assignment,19.39,100,19.39,Submitted,None,None,0,0,0,1,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
5,1006,210,210_A1,Assignment 1,Assignment,59.38,100,59.38,Submitted,None,None,0,0,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
6,1007,210,210_A1,Assignment 1,Assignment,78.12,100,78.12,Submitted,None,None,0,0,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
7,1008,210,210_A1,Assignment 1,Assignment,57.97,100,57.97,Submitted,None,None,0,0,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
8,1009,210,210_A1,Assignment 1,Assignment,58.87,100,58.87,Submitted,None,None,0,0,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11
9,1010,210,210_A1,Assignment 1,Assignment,75.28,100,75.28,Late,None,None,9,1,0,0,fallback_from_cohort,2026-05-11T00:14:16.202080+00:00,2026-05-11


In [14]:
dim_course.head()

,course_id,course_shortname,course_name
0,210,SC-1,Sample Course 1
1,211,SC-2,Sample Course 2


In [15]:
dim_student.head()

,student_id,username,first_name,last_name,full_name,email,role
0,1001,isabella.dummy1001,Isabella,Dummy,Isabella Dummy,isabella.dummy1001@sandbox.spi.nsw.edu.au,student
1,1002,harper.dummy1002,Harper,Dummy,Harper Dummy,harper.dummy1002@sandbox.spi.nsw.edu.au,student
2,1003,logan.dummy1003,Logan,Dummy,Logan Dummy,logan.dummy1003@sandbox.spi.nsw.edu.au,student
3,1004,amelia.dummy1004,Amelia,Dummy,Amelia Dummy,amelia.dummy1004@sandbox.spi.nsw.edu.au,student
4,1005,ethan.dummy1005,Ethan,Dummy,Ethan Dummy,ethan.dummy1005@sandbox.spi.nsw.edu.au,student


In [16]:
cohort_summary = pd.DataFrame([
    {"dataset": "dim_student", "rows": len(dim_student)},
    {"dataset": "dim_course", "rows": len(dim_course)},
    {"dataset": "dim_site", "rows": len(dim_site)},
    {"dataset": "dim_date", "rows": len(dim_date)},
    {"dataset": "fact_student_risk", "rows": len(fact_student_risk)},
    {"dataset": "fact_assessment", "rows": len(fact_assessment)},
    {"dataset": "dim_assessment", "rows": len(dim_assessment)},
])

cohort_summary

,dataset,rows
0,dim_student,22
1,dim_course,2
2,dim_site,1
3,dim_date,1
4,fact_student_risk,44
5,fact_assessment,132
6,dim_assessment,6
